In [1]:
import pandas as pd
import numpy as np

In [ ]:
def clean_noise_data_pandas(df: pd.DataFrame) -> pd.DataFrame:
    """
    Làm sạch dữ liệu: xóa trùng lặp, xóa NULL, lỗi logic và loại bỏ giá trị ngoại lai (Outliers)
    """
    initial_rows = len(df)
    print(f"[*] Bắt đầu xử lý nhiễu. Số dòng ban đầu: {initial_rows}")

    # 1. Xóa dòng trùng lặp 100%
    df_clean = df.drop_duplicates()

    # 2. Xóa các dòng chứa NULL ở các cột cốt lõi
    critical_cols = ['Date', 'Sales_Amount', 'Quantity']
    df_clean = df_clean.dropna(subset=critical_cols)

    # 3. Lọc nhiễu logic (Giao dịch hoàn tiền, lỗi nhập số lượng 0)
    df_clean = df_clean[(df_clean['Sales_Amount'] > 0) & (df_clean['Quantity'] > 0)]

    # 4. Xử lý Outlier (Giá trị ngoại lai) cho cột Quantity bằng IQR
    Q1 = df_clean['Quantity'].quantile(0.25)
    Q3 = df_clean['Quantity'].quantile(0.75)
    IQR = Q3 - Q1
    upper_bound = Q3 + 1.5 * IQR
    
    # Giữ lại các dòng có Quantity hợp lý
    df_clean = df_clean[df_clean['Quantity'] <= upper_bound]

    final_rows = len(df_clean)
    print(f"[*] Xử lý hoàn tất. Đã loại bỏ: {initial_rows - final_rows} dòng nhiễu.\n")
    
    # Trả về bản copy để tránh lỗi SettingWithCopyWarning ở các bước sau
    return df_clean.copy()

In [ ]:
def etl_pipeline(file_path):
    df_raw = pd.read_csv(file_path)
    
    # 2. TRANSFORM - BƯỚC 1: Xử lý cột 
    df = clean_noise_data_pandas(df_raw)
    # Xóa cột Transaction_ID vì độ phân tán 100%
    if 'Transaction_ID' in df.columns:
        df = df.drop(columns=['Transaction_ID'])
        
    # Chuyển đổi Date sang định dạng YYYYMM (ví dụ: 3/7/2025 -> 202503)
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    df['Date'] = df['Date'].dt.strftime('%Y%m')
    
    # Xác định đâu là Dimension (cột phân loại) và Measure (cột đo lường)
    measures = ['Sales_Amount', 'Quantity']
    dimensions = [col for col in df.columns if col not in measures]
    
    # 3. TRANSFORM - BƯỚC 2: Dimension Ordering
    # Tính Cardinality (số lượng giá trị unique) cho từng Dimension
    cardinalities = {col: df[col].nunique() for col in dimensions}
    
    # Sắp xếp các Dimension theo Cardinality từ thấp đến cao
    sorted_dimensions = sorted(cardinalities.keys(), key=lambda k: cardinalities[k])
    
    # 4. TRANSFORM - BƯỚC 3: Integer Encoding
    mapping_dict = {}
    for col in sorted_dimensions:
        unique_vals = sorted(df[col].dropna().unique())
        
        # Tạo dictionary map: Giá trị String -> Integer
        col_mapping = {val: idx + 1 for idx, val in enumerate(unique_vals)}
        mapping_dict[col] = col_mapping
        
        # Apply mapping vào DataFrame
        df[col] = df[col].map(col_mapping)
        
    # Tổ chức lại thứ tự cột: Sorted Dimensions đứng trước, Measures đứng sau
    final_columns = sorted_dimensions + measures
    df_final = df[final_columns]
    
    # 5. LOAD: Chuyển đổi thành Numpy 2D Array (List of Lists)
    processed_data = df_final.to_numpy()
    
    return processed_data, mapping_dict, final_columns

In [ ]:
file_csv = 'pos_data.csv'
processed_array, mappings, col_names = etl_pipeline(file_csv)

print("Thứ tự cột sau khi sắp xếp (Cardinality thấp -> cao):")
print(col_names)

print("\nMảng 2D (2 dòng đầu tiên):")
print(processed_array[:2])

print("\nMapping Dictionary (Ví dụ Region):")
print(mappings)

Thứ tự cột sau khi sắp xếp (Cardinality thấp -> cao):
['Customer_Type', 'Region', 'Payment_Method', 'Category', 'City', 'Date', 'Sales_Amount', 'Quantity']

Mảng 2D (2 dòng đầu tiên):
[[       1        1        4        2        9        2   327000        2]
 [       2        2        1        1        3       15 15950000        1]]

Mapping Dictionary (Ví dụ Region):
{'Customer_Type': {'Normal': 1, 'VIP': 2}, 'Region': {'Miền Bắc': 1, 'Miền Nam': 2, 'Miền Trung': 3}, 'Payment_Method': {'Bank Transfer': 1, 'Cash': 2, 'Credit Card': 3, 'E-Wallet': 4}, 'Category': {'Electronics': 1, 'F&B': 2, 'Fashion': 3, 'Grocery': 4, 'Home Appliances': 5}, 'City': {'Bình Dương': 1, 'Bắc Ninh': 2, 'Cần Thơ': 3, 'Huế': 4, 'Hà Nội': 5, 'Hải Phòng': 6, 'Nghệ An': 7, 'Nha Trang': 8, 'Quảng Ninh': 9, 'TP.HCM': 10, 'Đà Nẵng': 11, 'Đồng Nai': 12}, 'Date': {'202502': 1, '202503': 2, '202504': 3, '202505': 4, '202506': 5, '202507': 6, '202509': 7, '202510': 8, '202511': 9, '202601': 10, '202602': 11, '202604': 

In [8]:
print(processed_array)

[[       1        1        4        2        9        2   327000        2]
 [       2        2        1        1        3       15 15950000        1]
 [       1        2        1        3        1       12   710000        2]
 [       1        2        3        4       10        9   287000        3]
 [       1        3        1        4        8        9   603000        6]
 [       1        2        4        3        3       17  1407000        2]
 [       1        1        4        3        5        2   561000        2]
 [       1        1        3        5        5       13  5906000        1]
 [       1        2        4        5       10        4 25000000        2]
 [       1        2        3        3       10        2   428000        1]
 [       2        2        3        3        1       10   203000        1]
 [       1        2        2        2        1       20   313000        3]
 [       1        2        4        2        1       14   145000        4]
 [       1        1      